In [0]:
%run "../00. Common/Config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"


In [0]:
races_df = spark.table(bronze_table)

In [0]:
display(races_df)

In [0]:
races_selected_df = races_df.select(
    "season",
    "round",
    "raceName",
    "date",
    "circuitID",
    "ingestion_timestamp",
    "source_file"
)

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "raceName": "race_name",
        })
)

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])

In [0]:
display(races_distinct_df)

In [0]:
import pyspark.sql.functions as F
races_final_df = (
    races_distinct_df
        .withColumn('race_name',F.initcap("race_name"))
)

In [0]:
display(races_final_df)

In [0]:
(
    races_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))